# EUDR TESSERA Training — Kaggle GPU

Trains two models:
- **TESSERA head** — frozen ResNet50 encoder + lightweight segmentation head (GPU)
- **TESSERA-embed head** — 100K-param conv head on precomputed GeoTESSERA 128-dim embeddings (CPU)

**GPU:** T4 x2 or P100  
**Runtime:** ~45 min total (TESSERA head ~30 min GPU, embed head ~10 min CPU)

### Before running:
Add these Kaggle Secrets (*Settings → Secrets*):
- `CDSE_USER` / `CDSE_PASSWORD` — Copernicus Data Space
- `GEE_PROJECT_ID` / `GEE_SERVICE_ACCOUNT_KEY` — Google Earth Engine (base64 JSON)

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo
import subprocess, os

REPO_URL = "https://github.com/rajul-kk/EUDR-compliance.git"  # update if needed
WORK_DIR = "/kaggle/working/eudr"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt
!pip install -q geotessera

In [ ]:
# Load credentials
import base64, os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["CDSE_USER"]      = secrets.get_secret("CDSE_USER")
os.environ["CDSE_PASSWORD"]  = secrets.get_secret("CDSE_PASSWORD")
os.environ["GEE_PROJECT_ID"] = secrets.get_secret("GEE_PROJECT_ID")

gee_key_b64 = secrets.get_secret("GEE_SERVICE_ACCOUNT_KEY")
gee_key_path = "/kaggle/working/gee_key.json"
with open(gee_key_path, "w") as f:
    f.write(base64.b64decode(gee_key_b64).decode())
os.environ["GEE_SERVICE_ACCOUNT_KEY_PATH"] = gee_key_path
print("Credentials loaded.")

In [ ]:
# Create directories
dirs = [
    "inputs", "models", "reports",
    "data/raw_satellite/2020_baseline",
    "data/raw_satellite/2024_current",
    "data/hybrid_masks",
    "data/embeddings",
    "data/geotessera_tile_masks",
    "data/predictions_2024_tessera",
    "data/predictions_2024_tessera-embed",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

In [ ]:
# Step 1: Farm discovery
if not os.path.exists("inputs/farms_osm.csv") or os.path.getsize("inputs/farms_osm.csv") < 100:
    !python src/find_farms.py
else:
    import pandas as pd
    n = len(pd.read_csv("inputs/farms_osm.csv"))
    print(f"Using existing farms_osm.csv ({n} farms)")

In [ ]:
# Step 2: Download Sentinel-2 composites (both years needed for TESSERA training)
!python src/sentinel_client.py \
    --csv-path inputs/farms_osm.csv \
    --year 2020 \
    --output-dir data/raw_satellite/2020_baseline

!python src/sentinel_client.py \
    --csv-path inputs/farms_osm.csv \
    --year 2024 \
    --output-dir data/raw_satellite/2024_current

In [ ]:
# Step 3: Generate hybrid masks from GEE
!python GEE_dynamic/src/generate_hybrid.py

## Part A — TESSERA Head Training (GPU)
Frozen ResNet50 encoder + trainable segmentation head on raw Sentinel imagery.
Uses AMP + DataParallel automatically.

In [ ]:
!python src/tessera_train.py \
    --raw-dir data/raw_satellite/2020_baseline \
    --mask-dir data/hybrid_masks \
    --output-model-path models/farm_tessera.pth \
    --epochs 10 \
    --batch-size 8 \
    --learning-rate 1e-3 \
    --val-ratio 0.15 \
    --patience 3 \
    --num-workers 4 \
    --seed 42

In [ ]:
# Run TESSERA inference
!python src/inference.py \
    --model-path models/farm_tessera.pth \
    --input-dir data/raw_satellite/2024_current \
    --output-dir data/predictions_2024_tessera \
    --model-type tessera

## Part B — TESSERA-Embed Head Training (CPU)
Downloads precomputed GeoTESSERA 128-dim embeddings, builds tile masks, trains 100K-param head on CPU.
GPU overhead exceeds compute for this tiny model — CPU is faster.

In [ ]:
# Download precomputed embeddings — bbox auto-derived from farms CSV
!python src/tessera_embedding_generation.py \
    --farms-csv inputs/farms_osm.csv \
    --year 2024 \
    --output-dir data/embeddings \
    --bbox-padding 0.5

In [ ]:
# Build tile-aligned masks from GEE hybrid masks
!python src/geotessera_mask_tiler.py \
    --tile-tiff-dir data/embeddings/global_0.1_degree_tiff_all \
    --source-mask-dir data/hybrid_masks \
    --out-dir data/geotessera_tile_masks \
    --year 2024 \
    --summary-json reports/geotessera_mask_tiling_summary.json

In [ ]:
# Train embedding head on CPU (intentional — model is 100K params)
!python src/tessera_embed_train.py \
    --embeddings-dir data/embeddings/global_0.1_degree_representation \
    --mask-dir data/geotessera_tile_masks \
    --output-model-path models/farm_tessera_embed_head.pth \
    --dataset-mode geotessera \
    --year 2024 \
    --epochs 10 \
    --batch-size 16 \
    --learning-rate 1e-3 \
    --patience 3 \
    --device cpu \
    --split-manifest-path reports/tessera_embed_split.json

In [ ]:
# Run embedding inference on 2024 embeddings
!python src/tessera_embed_infer.py \
    --model-path models/farm_tessera_embed_head.pth \
    --embeddings-dir data/embeddings/global_0.1_degree_representation \
    --output-dir data/predictions_2024_tessera-embed \
    --year 2024 \
    --reference-image-dir data/embeddings/global_0.1_degree_tiff_all

In [ ]:
# Compare model outputs — tessera vs tessera-embed predictions
import pandas as pd, glob, os

for model_name, pred_dir in [
    ("tessera", "data/predictions_2024_tessera"),
    ("tessera-embed", "data/predictions_2024_tessera-embed"),
]:
    tifs = glob.glob(os.path.join(pred_dir, "*.tif"))
    print(f"{model_name}: {len(tifs)} prediction masks")

In [ ]:
# Package all models and reports for download
!zip -r /kaggle/working/eudr_tessera_outputs.zip \
    models/farm_tessera.pth \
    models/farm_tessera_embed_head.pth \
    reports/geotessera_mask_tiling_summary.json \
    reports/tessera_embed_split.json

print("Download eudr_tessera_outputs.zip from the Output panel.")